# ⚡ EduVault Content Delivery System (CDS) — Cloud Supercharger
### High-Speed Automated Batch Ingestion & Cloud Transcoding Engine

> **Performance Notice:** Running on Google Colab / Cloud VPS provides a **1.5 to 2.5 Gbps network interface** with direct European peering to Telegram Data Centers (DC2/DC4). Upload speeds consistently achieve **45 to 80 MB/s**, completing a 970-lecture batch in **~5 hours** instead of 3 days on residential broadband.

---
### Architecture Overview
- **Ingress CDN:** Native Linux `N_m3u8DL-RE` with binary fragment merging and automated retry.
- **Egress MTProto:** Pyrogram 4-Worker Turbo Upload with C-accelerated `tgcrypto` AES and warm session pooling.
- **State Persistence:** Connected directly to **Supabase Cloud PostgreSQL** (`video_assets` table). All progress, checkpoints, and completed lecture registrations persist permanently across runtime resets.

## Step 1 · One-Click System & Binary Auto-Provisioner
Installs `ffmpeg`, `aria2`, downloads native Linux `N_m3u8DL-RE`, builds `tgcrypto`, and installs repository requirements.

In [ ]:
# Clone or pull latest CDS repository if not already in workspace
import os, sys
if not os.path.exists("scripts"):
    print("Cloning repository...")
    !git clone --depth 1 https://github.com/ArYaN-X404/CDS.git . || true

# Execute the automated cloud provisioner
!python scripts/setup_cloud.py

## Step 2 · Credential Injection (Colab Secrets / Environment)
Securely retrieves API keys from Google Colab Secrets (`userdata`), `.env`, or interactive input.

In [ ]:
import os
import getpass

# Try loading from Colab Secrets first
try:
    from google.colab import userdata
    get_secret = lambda k: userdata.get(k)
except Exception:
    get_secret = lambda k: os.environ.get(k)

def resolve_credential(key_name, prompt_msg, default=""):
    val = None
    try:
        val = get_secret(key_name)
    except Exception:
        pass
    if not val:
        val = os.environ.get(key_name)
    if not val and not default:
        val = getpass.getpass(f"Enter {key_name} ({prompt_msg}): ")
    return val or default

# Set critical environment variables
os.environ["API_ID"] = str(resolve_credential("API_ID", "Telegram API ID"))
os.environ["API_HASH"] = str(resolve_credential("API_HASH", "Telegram API Hash"))
os.environ["BOT_TOKEN"] = str(resolve_credential("BOT_TOKEN", "Primary Telegram Bot Token"))
os.environ["CHANNEL_ID"] = str(resolve_credential("CHANNEL_ID", "Telegram Storage Channel ID", "-1004460495677"))
os.environ["OWNER_ID"] = str(resolve_credential("OWNER_ID", "Owner Telegram User ID", "7507697121"))
os.environ["FAST_UPLOAD_WORKERS"] = str(resolve_credential("FAST_UPLOAD_WORKERS", "Pyrogram Upload Workers", "14"))

# Supabase Cloud credentials (ensures zero data loss across Colab disconnects)
os.environ["SUPABASE_URL"] = str(resolve_credential("SUPABASE_URL", "Supabase URL", "https://fytemrmebqovsclumwqu.supabase.co"))
os.environ["SUPABASE_KEY"] = str(resolve_credential("SUPABASE_KEY", "Supabase Service Role Key", ""))

print("\n✅ All environment credentials securely loaded into runtime environment!")

## Step 3 · Diagnostic Verification (Supabase & Database Telemetry)
Runs instant SQLite and Supabase Cloud REST diagnostic to confirm connectivity and check current progress.

In [ ]:
!python scripts/fast_codebase_tool.py diag

## Step 4 · Background Keepalive & Disconnect Guard
Prevents Google Colab from timing out after 90 minutes of inactive user input.

In [ ]:
import time, threading, sys

def colab_keepalive_worker():
    while True:
        time.sleep(60)
        sys.stdout.write("\r[HEARTBEAT] Pipeline active. Uplink open. \n")
        sys.stdout.flush()

t = threading.Thread(target=colab_keepalive_worker, daemon=True)
t.start()
print("🛡️ Colab Keepalive Daemon active in background.")

## Step 5 · Launch Ingestion Pipeline
Launch the Telegram Central Bot or execute `cloud_worker.py` to stream live telemetry to local monitor.

In [ ]:
# Run Remote Cloud Worker Agent (Streams live telemetry to Local Monitor)
!python scripts/cloud_worker.py